In [1]:
# %%
# ============================================
# Section 1. Imports, paths, setup
# Conceptually:
# - A: epoch-level bandpower features
# - B: epoch-level wPLI edge-vector features (computed via sliding windows within each epoch)
# - Compare 3 models: A, B, A+B
# - Subject-aware nested CV with RF grid search
# ============================================

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / "data"
DERIVED_ROOT = DATA_ROOT / "derived"

FEAT_A_PATH = DERIVED_ROOT / "features" / "features_A_bandpower_epochwise.csv"
FEAT_B_PATH = DERIVED_ROOT / "features" / "features_B_wpli_edges_epoch_sliding.csv"  # <- NEW (epoch-level via sliding windows)

OUT_DIR = DERIVED_ROOT / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUT_DIR / "results_rf_nested_groupkfold_Aepoch_BwpliEpochSliding.csv"
PRED_PATH    = OUT_DIR / "predictions_rf_nested_groupkfold_Aepoch_BwpliEpochSliding.csv"
PARAMS_PATH  = OUT_DIR / "best_params_rf_nested_groupkfold_Aepoch_BwpliEpochSliding.csv"

print("A:", FEAT_A_PATH)
print("B:", FEAT_B_PATH)
print("OUT:", OUT_DIR)

A: /srv/scratch/z5718315/ketamine-project/data/derived/features/features_A_bandpower_epochwise.csv
B: /srv/scratch/z5718315/ketamine-project/data/derived/features/features_B_wpli_edges_epoch_sliding.csv
OUT: /srv/scratch/z5718315/ketamine-project/data/derived/models


In [2]:
# %%
# ============================================
# Section 2. Load A and B and align at the epoch level
# Merge on (subject_id, recording_number, epoch_index_original)
# Conceptually:
# - Both A and B should now have one row per epoch.
# - We keep label only once (from A).
# ============================================

A = pd.read_csv(FEAT_A_PATH)
B = pd.read_csv(FEAT_B_PATH)

A_ok = A[A.get("extract_ok", True) == True].copy()
B_ok = B[B.get("extract_ok", True) == True].copy()

key_cols = ["subject_id", "recording_number", "epoch_index_original"]

for c in key_cols + ["drug"]:
    assert c in A_ok.columns, f"Missing in A: {c}"
    assert c in B_ok.columns, f"Missing in B: {c}"

# Keep label only once from A
A_keep = A_ok.copy()
B_keep = B_ok.drop(columns=["drug"], errors="ignore").copy()

# Prefix non-key columns
A_ren = {c: f"A__{c}" for c in A_keep.columns if c not in key_cols}
B_ren = {c: f"B__{c}" for c in B_keep.columns if c not in key_cols}

A_keep = A_keep.rename(columns=A_ren)
B_keep = B_keep.rename(columns=B_ren)

df = A_keep.merge(B_keep, on=key_cols, how="inner")

assert "A__drug" in df.columns, "Expected A__drug after renaming"
df["drug"] = df["A__drug"]

print("Merged rows (epochs):", len(df))
print("Subjects:", df["subject_id"].nunique())
print("Recordings:", df[["subject_id", "recording_number"]].drop_duplicates().shape[0])

display(df.groupby(["subject_id", "drug"]).size().unstack(fill_value=0).head())

Merged rows (epochs): 276
Subjects: 10
Recordings: 20


drug,awake,ketamine
subject_id,,
210,14,11
219,14,16
249,15,17
251,14,13
265,13,11


In [3]:
# %%
# ============================================
# Section 3. Define target, groups, feature matrices
# Conceptually:
# - y: ketamine vs no-ketamine
# - groups: subject_id for subject-aware splitting
# - XA: bandpower epoch features
# - XB: wPLI edge-vector features (epoch-level via sliding windows)
# - XAB: concatenation
# ============================================

y = (df["drug"] == "ketamine").astype(int).to_numpy()
groups = df["subject_id"].astype(str).to_numpy()

# Exclude anything that could be metadata / leakage
EXCLUDE_EXACT = {
    "A__extract_ok", "B__extract_ok",
    "A__sfreq", "B__sfreq",
    "A__n_channels", "B__n_channels",
    "A__epoch_len_sec", "B__epoch_len_sec",
    "A__n_epochs_before", "B__n_epochs_before",
    "A__n_epochs_after", "B__n_epochs_after",
    "B__n_windows", "B__win_len_sec", "B__win_step_sec",  # window meta
}
EXCLUDE_CONTAINS = ["file_path", "extract_error", "drug", "eyes", "subject_id", "recording_number", "epoch_index"]

def select_feature_cols(prefix: str) -> list[str]:
    cols = []
    for c in df.columns:
        if not c.startswith(prefix):
            continue
        if c in EXCLUDE_EXACT:
            continue
        if any(x in c for x in EXCLUDE_CONTAINS):
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

A_feat_cols = select_feature_cols("A__")
B_feat_cols = select_feature_cols("B__")

assert len(A_feat_cols) > 0, "No A features found"
assert len(B_feat_cols) > 0, "No B features found"

XA = df[A_feat_cols].to_numpy()
XB = df[B_feat_cols].to_numpy()
XAB = np.concatenate([XA, XB], axis=1)

print("n A features:", len(A_feat_cols), "XA:", XA.shape)
print("n B features:", len(B_feat_cols), "XB:", XB.shape)
print("n A+B features:", XAB.shape[1], "XAB:", XAB.shape)

print("Class balance (epochs): ketamine=1:", int(y.sum()), "no-ketamine=0:", int((1-y).sum()))
print("Unique subjects:", len(np.unique(groups)))

n A features: 26 XA: (276, 26)
n B features: 5674 XB: (276, 5674)
n A+B features: 5700 XAB: (276, 5700)
Class balance (epochs): ketamine=1: 135 no-ketamine=0: 141
Unique subjects: 10


In [ ]:
# %%
# ============================================
# Section 4. Nested subject-aware RF (hyperparameter tuned)
# Variant 1 (NO leakage): PCA is fit inside CV via Pipeline (for B and A+B)
# Parallel: across (model × outer fold) tasks, no oversubscription
# ============================================

from __future__ import annotations

import os
import re
import subprocess
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd

from joblib import Parallel, delayed
from scipy.stats import randint
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline


# ----------------------------
# Robust core detection (PBS may not export PBS_NP)
# ----------------------------
def get_pbs_ncpus(default: int = 8) -> int:
    jobid = os.environ.get("PBS_JOBID")
    if not jobid:
        return default
    try:
        out = subprocess.check_output(["qstat", "-f", jobid], text=True)
        m = re.search(r"Resource_List\.ncpus\s*=\s*(\d+)", out)
        return int(m.group(1)) if m else default
    except Exception:
        return default


# ----------------------------
# Config
# ----------------------------
OUTER_SPLITS = 5
INNER_SPLITS = 4

# Search budget (per outer fold). Dev: 10–30, Solid: 50–120, Final: 150–250
N_ITER = 10

# PCA dimension for B / A+B. Start ~300–800 for high-dim edge vectors.
PCA_N_COMPONENTS = 100

SEED = 0

N_CORES = get_pbs_ncpus(default=8)
print("Detected N_CORES =", N_CORES)

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)

# Define which models should use PCA
USE_PCA_FOR = {"B_wpli_edges_epoch_sliding", "C_AplusB_epoch_sliding"}

# Param distributions:
# - For PCA pipelines, prefix RF params with "rf__"
# - For non-PCA models, use plain RF params
param_dist_rf = {
    "n_estimators": randint(200, 801),
    "max_depth": [None, 10, 20],
    "min_samples_split": randint(2, 11),
    "min_samples_leaf": randint(1, 5),
    "max_features": ["sqrt", 0.05, 0.1, 0.2],
}

param_dist_pca_rf = {f"rf__{k}": v for k, v in param_dist_rf.items()}
# Optional: you can also tune PCA dims cheaply by uncommenting:
# param_dist_pca_rf["pca__n_components"] = [200, 300, 500, 800]


@dataclass(frozen=True)
class ModelSpec:
    X: np.ndarray
    name: str


MODEL_SPECS = [
    ModelSpec(XA, "A_bandpower_epoch"),
    ModelSpec(XB, "B_wpli_edges_epoch_sliding"),
    ModelSpec(XAB, "C_AplusB_epoch_sliding"),
]

# We parallelize at task level: (model × outer fold)
N_TASKS = len(MODEL_SPECS) * OUTER_SPLITS  # 3*5=15
N_JOBS = min(N_CORES, N_TASKS)


def _make_estimator_and_space(model_name: str) -> tuple[Any, dict[str, Any]]:
    """
    Returns (estimator, param_distributions) for RandomizedSearchCV.
    PCA is applied inside CV for specified models (no leakage).
    """
    if model_name in USE_PCA_FOR:
        pca = PCA(
            n_components=PCA_N_COMPONENTS,
            svd_solver="randomized",
            random_state=SEED,
        )
        rf = RandomForestClassifier(
            random_state=SEED,
            class_weight="balanced_subsample",
            n_jobs=1,  # avoid nested parallelism
        )
        est = Pipeline([
            ("pca", pca),
            ("rf", rf),
        ])
        return est, param_dist_pca_rf

    # No PCA for model A by default
    rf = RandomForestClassifier(
        random_state=SEED,
        class_weight="balanced_subsample",
        n_jobs=1,  # avoid nested parallelism
    )
    return rf, param_dist_rf


def _fit_eval_one_task(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    model_name: str,
    fold: int,
    tr: np.ndarray,
    te: np.ndarray,
    scoring: str,
) -> tuple[dict[str, Any], list[dict[str, Any]], dict[str, Any]]:
    """
    One task = one outer fold for one model variant.
    Inner tuning uses RandomizedSearchCV.
    PCA (when enabled) is fit ONLY on training folds via Pipeline.
    """
    Xtr, Xte = X[tr], X[te]
    ytr, yte = y[tr], y[te]
    gtr, gte = groups[tr], groups[te]

    inner_cv = GroupKFold(n_splits=INNER_SPLITS)

    estimator, space = _make_estimator_and_space(model_name)

    rs = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=space,
        n_iter=N_ITER,
        cv=inner_cv,
        scoring=scoring,
        n_jobs=1,           # single-threaded inside each joblib worker
        refit=True,
        random_state=SEED + fold,  # vary per fold deterministically
        verbose=0,
    )

    rs.fit(Xtr, ytr, groups=gtr)
    best = rs.best_estimator_

    # --- Params row ---
    param_row: dict[str, Any] = {
        "model": model_name,
        "fold": fold,
        "n_iter": int(N_ITER),
        "best_score_inner": float(rs.best_score_),
        **rs.best_params_,
    }
    # Record PCA config for clarity (even if not tuned)
    if model_name in USE_PCA_FOR:
        param_row["pca_n_components"] = int(PCA_N_COMPONENTS)

    # --- Evaluate on outer test ---
    proba = best.predict_proba(Xte)[:, 1]
    yhat = (proba >= 0.5).astype(int)

    acc = float(accuracy_score(yte, yhat))
    bacc = float(balanced_accuracy_score(yte, yhat))
    try:
        aucv = float(roc_auc_score(yte, proba))
    except Exception:
        aucv = np.nan

    cm = confusion_matrix(yte, yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (np.nan, np.nan, np.nan, np.nan)

    fold_row: dict[str, Any] = {
        "model": model_name,
        "fold": fold,
        "n_test": int(len(te)),
        "n_test_subjects": int(len(np.unique(gte))),
        "accuracy": acc,
        "balanced_accuracy": bacc,
        "roc_auc": aucv,
        "tn": float(tn), "fp": float(fp), "fn": float(fn), "tp": float(tp),
    }

    pred_rows: list[dict[str, Any]] = []
    for i in range(len(te)):
        pred_rows.append({
            "model": model_name,
            "fold": fold,
            "subject_id": gte[i],
            "y_true": int(yte[i]),
            "y_proba": float(proba[i]),
            "y_pred": int(yhat[i]),
        })

    return fold_row, pred_rows, param_row


def run_all_models_nested_rf(
    y: np.ndarray,
    groups: np.ndarray,
    scoring: str = "balanced_accuracy",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Runs all model variants with nested CV.
    Parallelizes across (model × outer fold) tasks.
    """
    splits = list(outer_cv.split(MODEL_SPECS[0].X, y, groups))

    tasks = []
    for spec in MODEL_SPECS:
        for fold, (tr, te) in enumerate(splits, start=1):
            tasks.append((spec.X, y, groups, spec.name, fold, tr, te, scoring))

    results = Parallel(n_jobs=N_JOBS, prefer="processes")(
        delayed(_fit_eval_one_task)(*t) for t in tasks
    )

    fold_rows = [r[0] for r in results]
    pred_rows = [pr for r in results for pr in r[1]]
    param_rows = [r[2] for r in results]

    folds_df = pd.DataFrame(fold_rows).sort_values(["model", "fold"]).reset_index(drop=True)
    preds_df = pd.DataFrame(pred_rows).sort_values(["model", "fold"]).reset_index(drop=True)
    params_df = pd.DataFrame(param_rows).sort_values(["model", "fold"]).reset_index(drop=True)

    return folds_df, preds_df, params_df


folds, preds, params = run_all_models_nested_rf(y=y, groups=groups)

display(folds)
display(params.head(10))

Detected N_CORES = 8


KeyboardInterrupt: 

In [ ]:
# %%
# ============================================
# Section 5. Save results
# ============================================

folds.to_csv(RESULTS_PATH, index=False)
preds.to_csv(PRED_PATH, index=False)
params.to_csv(PARAMS_PATH, index=False)

print("Saved results →", RESULTS_PATH)
print("Saved predictions →", PRED_PATH)
print("Saved params →", PARAMS_PATH)

NameError: name 'folds' is not defined

In [6]:
import socket
print("host:", socket.gethostname())
print("PBS_JOBID:", os.environ.get("PBS_JOBID"))
print("N_CORES:", N_CORES)

host: katana3
PBS_JOBID: None
N_CORES: 8
